In [2]:
"""
Barcode Damage Detection — Dataset Preparation (Fresh Start)
=============================================================

GROUND TRUTH STRATEGY
---------------------
PASS = clean barcode image — physically intact, clearly visible
FAIL = same image with synthetic physical damage applied

FAIL means ONLY physical damage:
  scratches, ink smudge, partial cover, print degradation,
  water damage, torn edge, missing section

NOT fail: blurry camera, barcode too far away, bad angle.
These are imaging problems, not barcode damage.

QUALITY FILTERING (PASS images only)
--------------------------------------
We reject source images where the barcode is too far away or blurry,
because those images are useless for training — the model cannot learn
damage patterns from a 5-pixel blurry barcode.

We use 3 checks on the raw image BEFORE resizing:
  1. Sharpness (Laplacian variance) — is the image in focus?
  2. Bar transitions — are barcode bars actually visible?
  3. Best-row transitions — even if barcode isn't centred, can we find it?

BARCODE RECOVERY
-----------------
For images where the barcode is small but still usable, we attempt to:
  1. Detect the barcode region using Sobel edge density
  2. Crop tightly around it
  3. Upscale the crop to 224×224 using bicubic interpolation
  4. Apply CLAHE contrast enhancement to make bars clearer

This recovers barcodes that would otherwise be rejected.
Only applied when barcode occupies < 60% of frame width.

Run:
    pip install datasets pillow opencv-python tqdm
    python prepare_dataset.py
"""

import os
import random
import shutil
import json
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
SEED         = 42
N_IMAGES     = 3000       # PASS images to collect → 3000 PASS + 3000 FAIL
IMG_SIZE     = (224, 224)
OUTPUT_DIR   = "barcode_dataset_v2"
SPLIT_RATIOS = (0.70, 0.15, 0.15)

# Quality filter threshold
# Calibrated against real images (at 224×224 as delivered by HuggingFace):
#   far_bottle   best_transitions=9  → REJECT
#   far_shampoo  best_transitions=6  → REJECT
#   good_box     best_transitions=43 → ACCEPT
#   good_parcel  best_transitions=54 → ACCEPT
#
# We use only best-row transitions — sharpness caused 91% rejection because
# dataset images are already 224×224 thumbnails with compressed sharpness scores.
MIN_BEST_TRANSITIONS = 10   # rejects only genuinely invisible barcodes

random.seed(SEED)
np.random.seed(SEED)


# ── Quality filter ────────────────────────────────────────────────────────────

def check_quality(pil_img):
    """
    Returns (passed: bool, reason: str).

    Single check: are barcode bars visible anywhere in the image?

    We scan every row and find the one with the most vertical edge content
    (the row that cuts through the most bar transitions). If even the best
    row has fewer than MIN_BEST_TRANSITIONS transitions, the barcode is
    genuinely too small or blurry to be useful for training.

    Why only one check:
      - Sharpness threshold caused 91% rejection — dataset images are already
        224×224 thumbnails with naturally compressed sharpness scores
      - Mid-row transitions fail when the barcode is off-centre vertically
      - Best-row transitions directly measures what we care about:
        are the bars actually visible somewhere in the image?

    Calibrated values (at 224×224):
      far_bottle   best_transitions=9  → REJECT
      far_shampoo  best_transitions=6  → REJECT
      good_box     best_transitions=43 → ACCEPT
      good_parcel  best_transitions=54 → ACCEPT
    """
    arr   = np.array(pil_img.convert("RGB"))
    gray  = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    sobel = np.uint8(np.absolute(cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)))

    # Find the row with the most vertical edge content
    best_row = int(sobel.sum(axis=1).argmax())
    best_t   = int(np.sum(
        np.abs(np.diff(gray[best_row, :].astype(int))) > 30))

    if best_t < MIN_BEST_TRANSITIONS:
        return False, f"barcode not visible (best_row_transitions={best_t} < {MIN_BEST_TRANSITIONS})"

    return True, "ok"


# ── Barcode crop + recovery ───────────────────────────────────────────────────

def extract_barcode(pil_img):
    """
    Attempts to find and return a tight crop of the barcode region.

    Strategy:
      1. Apply CLAHE to boost local contrast (helps with faded barcodes)
      2. Compute vertical Sobel edges (barcodes = many vertical bars)
      3. Morphologically close the edge map to merge nearby bar clusters
      4. Find the contour with highest edge density × area score
      5. If barcode is small in frame, upscale with bicubic before final resize

    If no clear barcode region is found, falls back to the whole image.
    Returns a 224×224 PIL image ready for the dataset.
    """
    arr  = np.array(pil_img.convert("RGB"))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape

    # Contrast enhancement
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    # Vertical edge map
    sobel  = np.uint8(np.absolute(
        cv2.Sobel(enhanced, cv2.CV_64F, 1, 0, ksize=3)))

    # Close gaps between bars
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 5))
    closed = cv2.morphologyEx(sobel, cv2.MORPH_CLOSE, kernel)
    _, thresh = cv2.threshold(closed, 50, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best_score = 0
    best_box   = None

    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        if cw * ch < w * h * 0.005:   # too small — noise
            continue
        aspect = cw / max(ch, 1)
        if aspect < 0.5:              # too tall — probably not a barcode
            continue
        # Score = edge density × relative size
        density = float(sobel[y:y+ch, x:x+cw].mean())
        score   = density * (cw * ch) / (w * h)
        if score > best_score:
            best_score = score
            best_box   = (x, y, cw, ch)

    if best_box is not None:
        x, y, cw, ch = best_box
        pad = 15
        x1  = max(0, x - pad);    y1 = max(0, y - pad)
        x2  = min(w, x + cw + pad); y2 = min(h, y + ch + pad)
        crop = arr[y1:y2, x1:x2]

        # If barcode is small in the frame, upscale before final resize
        # This preserves more bar detail than stretching a tiny crop directly
        crop_w = x2 - x1
        if crop_w < w * 0.4:          # barcode occupies < 40% of frame width
            scale  = max(2, int(224 / crop_w))
            crop   = cv2.resize(
                crop,
                (crop.shape[1] * scale, crop.shape[0] * scale),
                interpolation=cv2.INTER_CUBIC,
            )
        return Image.fromarray(crop).resize(IMG_SIZE, Image.LANCZOS)

    # Fallback — use the whole image
    return pil_img.convert("RGB").resize(IMG_SIZE, Image.LANCZOS)


# ── Damage functions ──────────────────────────────────────────────────────────
# Each simulates a real physical defect. Applied to PASS images to create FAIL.

def add_scratches(a):
    """Horizontal lines — pen marks, scuff marks."""
    h, w = a.shape[:2]
    for _ in range(random.randint(3, 8)):
        x1 = random.randint(0, w-1)
        x2 = random.randint(0, w-1)
        y  = random.randint(h//4, 3*h//4)
        thickness = random.randint(1, 4)
        color = random.choice([(0,0,0), (80,80,80), (200,200,200)])
        cv2.line(a, (x1, y), (x2, y), color, thickness)
    return a

def add_diagonal_scratches(a):
    """Diagonal scuff marks."""
    h, w = a.shape[:2]
    for _ in range(random.randint(2, 5)):
        x1, y1 = random.randint(0, w-1), random.randint(0, h-1)
        x2 = x1 + random.randint(-w//3, w//3)
        y2 = y1 + random.randint(-h//3, h//3)
        color = random.choice([(0,0,0), (60,60,60)])
        cv2.line(a, (x1,y1), (x2,y2), color, random.randint(1, 3))
    return a

def add_ink_smudge(a):
    """Dark elliptical blotch — ink smear or wet damage."""
    h, w = a.shape[:2]
    cx = random.randint(w//4, 3*w//4)
    cy = random.randint(h//4, 3*h//4)
    rw = random.randint(w//8, w//3)
    rh = random.randint(h//8, h//3)
    overlay = a.copy()
    cv2.ellipse(overlay, (cx, cy), (rw, rh), 0, 0, 360,
                (random.randint(0, 60), 0, 0), -1)
    alpha = random.uniform(0.4, 0.8)
    return cv2.addWeighted(overlay, alpha, a, 1-alpha, 0)

def add_partial_cover(a):
    """Rectangle covering part of bars — sticker or tape."""
    h, w = a.shape[:2]
    x_start = random.randint(w//5, w//2)
    cover_w = random.randint(w//8, w//4)
    color = random.choice([(255,255,255), (255,255,180), (200,200,200)])
    a[0:h, x_start:x_start+cover_w] = color
    return a

def add_print_degradation(a):
    """Faded vertical strips — ink running out."""
    h, w = a.shape[:2]
    for _ in range(random.randint(3, 8)):
        x      = random.randint(0, w-1)
        fade_w = random.randint(1, 3)
        amount = random.randint(80, 160)
        strip  = a[0:h, x:x+fade_w].astype(np.int32)
        a[0:h, x:x+fade_w] = np.clip(strip + amount, 0, 255).astype(np.uint8)
    return a

def add_water_damage(a):
    """Wavy distortion + dark spots — moisture damage."""
    h, w = a.shape[:2]
    map_x = np.zeros((h, w), dtype=np.float32)
    map_y = np.zeros((h, w), dtype=np.float32)
    amp   = random.randint(3, 8)
    freq  = random.uniform(0.05, 0.15)
    for i in range(h):
        for j in range(w):
            map_x[i, j] = j + amp * np.sin(2 * np.pi * i * freq)
            map_y[i, j] = i + amp * np.cos(2 * np.pi * j * freq)
    a = cv2.remap(a, map_x, map_y, cv2.INTER_LINEAR,
                  borderValue=(255, 255, 255))
    mask = np.random.random((h, w)) > 0.85
    a[mask] = np.clip(
        a[mask].astype(int) - random.randint(40, 80), 0, 255)
    return a.astype(np.uint8)

def add_torn_edge(a):
    """Jagged tear from an edge or corner — torn label."""
    h, w  = a.shape[:2]
    bg    = tuple(int(c) for c in
                  np.percentile(a[0:10, 0:10], 90, axis=(0,1)).tolist())
    style = random.choice(["corner", "edge", "bottom"])

    if style == "corner":
        corner  = random.choice(["tl", "tr", "bl", "br"])
        depth_x = random.randint(w//5, w//2)
        depth_y = random.randint(h//5, h//2)
        n       = random.randint(8, 16)
        if corner == "tr":
            xs = np.clip(np.linspace(w-depth_x, w, n).astype(int)
                         + np.random.randint(-8, 8, n), 0, w)
            ys = np.clip(np.linspace(0, depth_y, n).astype(int)
                         + np.random.randint(-8, 8, n), 0, h)
            pts = np.array(list(zip(xs, ys)) + [(w, 0)], dtype=np.int32)
        elif corner == "tl":
            xs = np.clip(np.linspace(0, depth_x, n).astype(int)
                         + np.random.randint(-8, 8, n), 0, w)
            ys = np.clip(np.linspace(depth_y, 0, n).astype(int)
                         + np.random.randint(-8, 8, n), 0, h)
            pts = np.array([(0, 0)] + list(zip(xs, ys)), dtype=np.int32)
        elif corner == "br":
            xs = np.clip(np.linspace(w, w-depth_x, n).astype(int)
                         + np.random.randint(-8, 8, n), 0, w)
            ys = np.clip(np.linspace(h-depth_y, h, n).astype(int)
                         + np.random.randint(-8, 8, n), 0, h)
            pts = np.array(list(zip(xs, ys)) + [(w, h)], dtype=np.int32)
        else:
            xs = np.clip(np.linspace(depth_x, 0, n).astype(int)
                         + np.random.randint(-8, 8, n), 0, w)
            ys = np.clip(np.linspace(h, h-depth_y, n).astype(int)
                         + np.random.randint(-8, 8, n), 0, h)
            pts = np.array([(0, h)] + list(zip(xs, ys)), dtype=np.int32)
        cv2.fillPoly(a, [pts.reshape(-1, 1, 2)], bg)

    elif style == "edge":
        side  = random.choice(["left", "right"])
        depth = random.randint(w//6, w//3)
        n     = random.randint(10, 20)
        ys    = np.linspace(0, h, n).astype(int)
        if side == "right":
            xs  = np.clip(w - depth + np.random.randint(-15, 15, n), w-depth-20, w)
            pts = np.array([(w, 0)] + list(zip(xs, ys)) + [(w, h)], dtype=np.int32)
        else:
            xs  = np.clip(depth + np.random.randint(-15, 15, n), 0, depth+20)
            pts = np.array([(0, 0)] + list(zip(xs, ys)) + [(0, h)], dtype=np.int32)
        cv2.fillPoly(a, [pts.reshape(-1, 1, 2)], bg)

    else:  # bottom peel
        depth = random.randint(h//5, h//2)
        n     = random.randint(10, 20)
        xs    = np.linspace(0, w, n).astype(int)
        ys    = np.clip(h - depth + np.random.randint(-12, 12, n),
                        h-depth-15, h)
        pts   = np.array([(0, h)] + list(zip(xs, ys)) + [(w, h)], dtype=np.int32)
        cv2.fillPoly(a, [pts.reshape(-1, 1, 2)], bg)

    return a

def add_missing_middle(a):
    """Band or blob where bars are absent — shattered or peeled print."""
    h, w  = a.shape[:2]
    style = random.choice(["h_band", "v_gap", "shatter"])

    if style == "h_band":
        band_h = random.randint(h//6, h//3)
        band_y = random.randint(h//4, h//2)
        fill   = random.choice([255, random.randint(160, 220)])
        a[band_y:band_y+band_h, :] = fill

    elif style == "v_gap":
        for _ in range(random.randint(1, 3)):
            gx = random.randint(w//5, 4*w//5)
            gw = random.randint(w//8, w//4)
            a[:, gx:gx+gw] = 255

    else:  # shatter
        for _ in range(random.randint(2, 5)):
            cx    = random.randint(w//4, 3*w//4)
            cy    = random.randint(h//4, 3*h//4)
            rw    = random.randint(w//8, w//3)
            rh    = random.randint(h//6, h//2)
            n     = random.randint(6, 10)
            ang   = np.linspace(0, 2*np.pi, n, endpoint=False)
            rv    = np.random.uniform(0.6, 1.0, n)
            px    = np.clip((cx + rw*rv*np.cos(ang)).astype(int), 0, w-1)
            py    = np.clip((cy + rh*rv*np.sin(ang)).astype(int), 0, h-1)
            pts   = np.stack([px, py], axis=1).reshape(-1, 1, 2)
            fill  = random.choice([255, 230, 200])
            cv2.fillPoly(a, [pts], (fill, fill, fill))

    return a


DAMAGE_FNS = [
    add_scratches,
    add_diagonal_scratches,
    add_ink_smudge,
    add_partial_cover,
    add_print_degradation,
    add_water_damage,
    add_torn_edge,
    add_missing_middle,
]

def apply_damage(pil_img):
    """Apply 1 or 2 random physical damage types to a clean image."""
    a       = np.array(pil_img.convert("RGB"))
    chosen  = random.sample(DAMAGE_FNS, k=random.randint(1, 2))
    for fn in chosen:
        a = fn(a)
    return Image.fromarray(a.astype(np.uint8))


# ── Main pipeline ─────────────────────────────────────────────────────────────

def main():
    from datasets import load_dataset

    tmp_pass = "tmp_pass_v2"
    tmp_fail = "tmp_fail_v2"
    for d in [tmp_pass, tmp_fail, OUTPUT_DIR]:
        os.makedirs(d, exist_ok=True)

    # ── Step 1: Stream and filter PASS images ─────────────────────────────────
    print("=" * 60)
    print("Step 1/5 — Collecting PASS images with quality filter...")
    print("=" * 60)
    print(f"  Target     : {N_IMAGES} images")
    print(f"  Filter     : best-row bar transitions > {MIN_BEST_TRANSITIONS}")
    print(f"               (rejects only truly invisible barcodes)")
    print(f"  Recovery   : barcode crop + upscale for small barcodes\n")

    ds = load_dataset(
        "amaye15/barcodes-google-ocr",
        split="train",
        streaming=True,
        columns=["pixel_values"],
    ).batch(1)

    collected  = 0
    scanned    = 0
    rejected   = 0
    reject_reasons = {}

    with tqdm(total=N_IMAGES, desc="  Collecting", unit=" imgs") as pbar:
        for batch in ds:
            scanned += 1
            try:
                raw_img = batch["pixel_values"][0]
                if not isinstance(raw_img, Image.Image):
                    raw_img = Image.fromarray(np.array(raw_img))
            except Exception:
                rejected += 1
                continue

            # Quality check on original (not yet resized)
            passed, reason = check_quality(raw_img)
            if not passed:
                rejected += 1
                key = reason.split("(")[0].strip()
                reject_reasons[key] = reject_reasons.get(key, 0) + 1
                continue

            # Extract and recover barcode region, then resize
            img = extract_barcode(raw_img)

            img.save(os.path.join(tmp_pass, f"pass_{collected:05d}.jpg"),
                     quality=92)
            del img, raw_img
            collected += 1
            pbar.update(1)
            pbar.set_postfix(scanned=scanned, rejected=rejected)

            if collected >= N_IMAGES:
                break

    print(f"\n  Scanned   : {scanned}")
    print(f"  Collected : {collected}  ({collected/scanned*100:.1f}% pass rate)")
    print(f"  Rejected  : {rejected}  ({rejected/scanned*100:.1f}%)")
    print(f"  Reasons   :")
    for r, c in sorted(reject_reasons.items(), key=lambda x: -x[1]):
        print(f"    {r:35s}: {c}")

    # ── Step 2: Generate FAIL images ──────────────────────────────────────────
    print("\n" + "="*60)
    print("Step 2/5 — Generating FAIL images (synthetic damage)...")
    print("="*60)

    pass_paths = sorted([
        os.path.join(tmp_pass, f) for f in os.listdir(tmp_pass)
    ])
    fail_count = 0

    with tqdm(total=N_IMAGES, desc="  Damaging", unit=" imgs") as pbar:
        for src_path in pass_paths:
            try:
                src = Image.open(src_path)
                dmg = apply_damage(src)
                src.close()
                dmg.save(
                    os.path.join(tmp_fail, f"fail_{fail_count:05d}.jpg"),
                    quality=92)
                dmg.close()
                fail_count += 1
                pbar.update(1)
            except Exception as e:
                print(f"  Warning: {e}")

    print(f"\n  Generated : {fail_count} FAIL images")

    # ── Step 3: Split ─────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("Step 3/5 — Building 70 / 15 / 15 splits...")
    print("="*60)

    all_records = (
        [(f, 1) for f in sorted(os.listdir(tmp_pass))[:N_IMAGES]] +
        [(f, 0) for f in sorted(os.listdir(tmp_fail))[:N_IMAGES]]
    )
    random.shuffle(all_records)

    n       = len(all_records)
    n_train = int(n * SPLIT_RATIOS[0])
    n_val   = int(n * SPLIT_RATIOS[1])
    splits  = {
        "train":      all_records[:n_train],
        "validation": all_records[n_train:n_train+n_val],
        "test":       all_records[n_train+n_val:],
    }

    # ── Step 4: Save ──────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("Step 4/5 — Saving to disk...")
    print("="*60)

    manifest = {}
    for split_name, rows in splits.items():
        split_dir = os.path.join(OUTPUT_DIR, split_name)
        os.makedirs(os.path.join(split_dir, "PASS"), exist_ok=True)
        os.makedirs(os.path.join(split_dir, "FAIL"), exist_ok=True)
        n_pass = n_fail = 0
        manifest[split_name] = []
        for i, (fname, label) in enumerate(
                tqdm(rows, desc=f"  {split_name:12s}")):
            src_dir    = tmp_pass if label == 1 else tmp_fail
            label_name = "PASS"   if label == 1 else "FAIL"
            dst = os.path.join(split_dir, label_name, f"{i:05d}.jpg")
            shutil.copy2(os.path.join(src_dir, fname), dst)
            manifest[split_name].append(
                {"file": dst, "label": label, "label_name": label_name})
            if label == 1: n_pass += 1
            else:          n_fail += 1
        print(f"    {split_name:12s}: {len(rows):5d}  "
              f"PASS={n_pass}  FAIL={n_fail}")

    with open(os.path.join(OUTPUT_DIR, "manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)

    shutil.rmtree(tmp_pass)
    shutil.rmtree(tmp_fail)

    print(f"\n  Saved to : {os.path.abspath(OUTPUT_DIR)}")

    # ── Step 5: Summary ───────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("DONE — load in PyTorch:")
    print("="*60)
    print("""
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader

    T = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])

    train_ds = datasets.ImageFolder("barcode_dataset_v2/train",      transform=T)
    val_ds   = datasets.ImageFolder("barcode_dataset_v2/validation",  transform=T)
    test_ds  = datasets.ImageFolder("barcode_dataset_v2/test",        transform=T)
    # train_ds.classes → ['FAIL', 'PASS']
    """)


if __name__ == "__main__":
    main()

Step 1/5 — Collecting PASS images with quality filter...
  Target     : 3000 images
  Filter     : best-row bar transitions > 10
               (rejects only truly invisible barcodes)
  Recovery   : barcode crop + upscale for small barcodes



Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

  Collecting: 100%|████████████████████████████████| 3000/3000 [07:12<00:00,  6.93 imgs/s, rejected=1445, scanned=4445]



  Scanned   : 4445
  Collected : 3000  (67.5% pass rate)
  Rejected  : 1445  (32.5%)
  Reasons   :
    barcode not visible                : 1445

Step 2/5 — Generating FAIL images (synthetic damage)...


...

    train       :  4200  PASS=2081  FAIL=2119


  validation  : 100%|███████████████████████████████████████████████████████████████| 900/900 [00:01<00:00, 550.54it/s]


    validation  :   900  PASS=462  FAIL=438


  test        : 100%|███████████████████████████████████████████████████████████████| 900/900 [00:01<00:00, 552.90it/s]


    test        :   900  PASS=457  FAIL=443

  Saved to : C:\Users\erum8\barcode_dataset_v2

DONE — load in PyTorch:

    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader

    T = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])

    train_ds = datasets.ImageFolder("barcode_dataset_v2/train",      transform=T)
    val_ds   = datasets.ImageFolder("barcode_dataset_v2/validation",  transform=T)
    test_ds  = datasets.ImageFolder("barcode_dataset_v2/test",        transform=T)
    # train_ds.classes → ['FAIL', 'PASS']
    
